In [1]:
import requests
import json
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

In [2]:
def safe_json(r, label=""):
    """Parsea JSON de una respuesta requests; imprime error y devuelve None si falla."""
    try:
        return r.json()
    except Exception:
        print(f"[ERROR json] {label} — HTTP {r.status_code} — body: {r.text[:300]!r}")
        return None


def load_geojson_jsonp(url):
    """Descarga un GeoJSON en formato JSONP y devuelve la lista de features."""
    response = requests.get(url)
    response.raise_for_status()
    raw = response.text.strip()
    if raw.startswith("jsonCallback("):
        raw = raw[len("jsonCallback("):]
        if raw.endswith(");"):
            raw = raw[:-2]
        elif raw.endswith(")"):
            raw = raw[:-1]
    return json.loads(raw)['features']


def features_to_df(features):
    """Convierte la lista de features GeoJSON a un DataFrame con las columnas de interés."""
    df = pd.DataFrame([{
        "nombre":      f['properties'].get('documentname'),
        "municipio":   f['properties'].get('municipality'),
        "provincia":   f['properties'].get('territory'),
        "latitud":     f['geometry']['coordinates'][1],
        "longitud":    f['geometry']['coordinates'][0],
        "type":        f['properties'].get('templatetype'),
        "tipo-comida": f['properties'].get('restorationtype'),
        "entorno":     f['properties'].get('marks'),
        "quality":     f['properties'].get('qualityassurance'),
        "euskal_sello": f['properties'].get('productclub'),
        "email":       f['properties'].get('tourismemail') or "",
        "web":         f['properties'].get('web') or "",
        "web_euskadi": f['properties'].get('friendlyurl') or ""
    } for f in features])
    # Normalizar URLs a https para evitar redirecciones
    df["web"] = df["web"].str.replace("http:", "https:", regex=False)
    df["web_euskadi"] = df["web_euskadi"].str.replace("http:", "https:", regex=False)
    return df


# --- Ejecución ---
URL_GEOJSON = "https://opendata.euskadi.eus/contenidos/ds_recursos_turisticos/restaurantes_sidrerias_bodegas/opendata/restaurantes-padding.geojson"

features = load_geojson_jsonp(URL_GEOJSON)
print(f"[OK] GeoJSON descargado: {len(features)} restaurantes")

df_opendata_restaurantes = features_to_df(features)



[OK] GeoJSON descargado: 716 restaurantes


In [3]:
# en los parametros de entrada debe pedir df1, df2, cat1, cat2

def merge_dataframes(df1, df2, cat1, cat2):
    """Une df1 (cat1) con df2 (cat2).
    Conserva todas las filas de df1 y añade de df2 solo las que no existan en df1
    (comparando por nombre + municipio)."""
    df1 = df1.copy()
    df2 = df2.copy()
    df1["categoria"] = cat1
    df2["categoria"] = cat2

    # Filas de df2 cuyo (nombre, municipio) NO está en df1
    key1 = set(zip(df1["nombre"], df1["municipio"]))
    nuevos = df2[~df2.apply(lambda r: (r["nombre"], r["municipio"]) in key1, axis=1)]

    if len(df2) - len(nuevos) > 0:
        print(f"Duplicados ignorados de {cat2}: {len(df2) - len(nuevos)}")

    result = pd.concat([df1, nuevos], ignore_index=True)
    print(f"Total tras merge: {len(result)} filas ({cat1}: {len(df1)}, {cat2}: {len(nuevos)})")
    return result

# df_opendata = merge_dataframes(df_opendata_bodegas, df_opendata_asador)


In [4]:
URL_GEOJSON_QUESERIA = "https://opendata.euskadi.eus/contenidos/ds_recursos_turisticos/queserias_conservera_productor/opendata/queserias-conserveras-productores-padding.geojson"
features_queseria = load_geojson_jsonp(URL_GEOJSON_QUESERIA)
df_opendata_queseria = features_to_df(features_queseria)
print(f"[OK] GeoJSON descargado: {len(features_queseria)} queserías")




[OK] GeoJSON descargado: 58 queserías


In [5]:
df_opendata = merge_dataframes(df_opendata_restaurantes, df_opendata_queseria, "restaurantes", "queseria")

Total tras merge: 774 filas (restaurantes: 716, queseria: 58)


In [6]:
URL_GEOJSON_BODEGA = "https://opendata.euskadi.eus/contenidos/ds_recursos_turisticos/bodegas_vino_txakoli/opendata/bodegas.geojson"
features_bodega = load_geojson_jsonp(URL_GEOJSON_BODEGA)
df_opendata_bodega = features_to_df(features_bodega)
print(f"[OK] GeoJSON descargado: {len(features_bodega)} bodegas")

[OK] GeoJSON descargado: 104 bodegas


In [7]:
df_opendata = merge_dataframes(df_opendata, df_opendata_bodega, "restaurantes", "bodegas")

Duplicados ignorados de bodegas: 104
Total tras merge: 774 filas (restaurantes: 774, bodegas: 0)


In [8]:
df_opendata


,nombre,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,quality,euskal_sello,email,web,web_euskadi,categoria
0,Agorregi,Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",None,None,agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
1,Aizian,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,None,None,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
2,Akelarre,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,None,1,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
3,Alameda,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,None,1,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
4,Almiketxu,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,None,1,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
769,Iruri,Legutio,Araba/Álava,42.980472,-2.642584,Tiendas,None,Montes y Valles vascos,None,None,bostibaietasc@gmail.com,https://euskadigastronomika.eus/es/iruri,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes
770,Conservas Arroyabe,Bermeo,Bizkaia,43.422108,-2.745513,Tiendas,None,Costa Vasca,None,None,info@arroyabe.es,https://www.arroyabe.es/,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes
771,Conservas Campos,Bermeo,Bizkaia,43.420379,-2.737545,Tiendas,None,Costa Vasca,None,None,,https://www.clubcampos.com/,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes
772,Burutxaga Hestebeteak,Amurrio,Araba/Álava,43.039381,-3.004257,Tiendas,None,Montes y Valles vascos,None,None,burutxaga.hestebeteak@gmail.com,,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes


In [9]:
# seleccionamos los distintos tipos de "tipo-comida"
print("Tipos de comida disponibles:")
df_opendata["tipo-comida"].unique()


Tipos de comida disponibles:


array(['Restaurante', 'Bodega Txakoli', 'Asador', 'Bodega', 'Sidreria',
       None, 'Trujal', 'Pastelería / Confitería', 'Comida Rápida'],
      dtype=object)

In [14]:
# eliminar las con tipo-comida "comida rápida"
df_opendata = df_opendata[df_opendata["tipo-comida"] != "Comida Rápida"].copy()


In [15]:
df_opendata.describe(include="all")


,nombre,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,quality,euskal_sello,email,web,web_euskadi,categoria,calidad
count,773,773,773,773.000000,773.000000,771,713,768,109,449,773,773,773,773,460
unique,762,174,5,NaN,NaN,2,7,11,1,1,604,610,773,1,2
top,Txinparta,Donostia / San Sebastián,Gipuzkoa,NaN,NaN,Restauración,Restaurante,Montes y Valles vascos,1,1,,,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,1
freq,2,59,369,NaN,NaN,713,475,338,109,449,165,158,1,773,362
mean,NaN,NaN,NaN,43.131393,-2.455138,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,0.255158,0.392681,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,42.488302,-3.386813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,43.034558,-2.799726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,43.253906,-2.501972,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,43.307750,-2.094860,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df_opendata.info()

<class 'pandas.core.frame.DataFrame'>
Index: 773 entries, 0 to 773
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   nombre        773 non-null    object 
 1   municipio     773 non-null    object 
 2   provincia     773 non-null    object 
 3   latitud       773 non-null    float64
 4   longitud      773 non-null    float64
 5   type          771 non-null    object 
 6   tipo-comida   713 non-null    object 
 7   entorno       768 non-null    object 
 8   quality       109 non-null    object 
 9   euskal_sello  449 non-null    object 
 10  email         773 non-null    object 
 11  web           773 non-null    object 
 12  web_euskadi   773 non-null    object 
 13  categoria     773 non-null    object 
 14  calidad       460 non-null    object 
dtypes: float64(2), object(13)
memory usage: 96.6+ KB


In [ ]:
# calidad = True si el lugar tiene al menos un sello de calidad o producto Euskal
def combinar_calidad(row):
    """Devuelve True si el lugar tiene algún sello de calidad o Euskal produktua."""
    return bool(row["quality"]) or bool(row["euskal_sello"])

df_opendata["calidad"] = df_opendata.apply(combinar_calidad, axis=1)


In [26]:
df_opendata

,nombre,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,quality,euskal_sello,email,web,web_euskadi,categoria,calidad
0,Agorregi,Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",False,False,agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False
1,Aizian,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,False,False,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False
2,Akelarre,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,False,1,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
3,Alameda,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,False,1,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
4,Almiketxu,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,False,1,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
769,Iruri,Legutio,Araba/Álava,42.980472,-2.642584,Tiendas,<NA>,Montes y Valles vascos,False,False,bostibaietasc@gmail.com,https://euskadigastronomika.eus/es/iruri,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False
770,Conservas Arroyabe,Bermeo,Bizkaia,43.422108,-2.745513,Tiendas,<NA>,Costa Vasca,False,False,info@arroyabe.es,https://www.arroyabe.es/,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False
771,Conservas Campos,Bermeo,Bizkaia,43.420379,-2.737545,Tiendas,<NA>,Costa Vasca,False,False,,https://www.clubcampos.com/,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False
772,Burutxaga Hestebeteak,Amurrio,Araba/Álava,43.039381,-3.004257,Tiendas,<NA>,Montes y Valles vascos,False,False,burutxaga.hestebeteak@gmail.com,,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False


In [27]:
df_opendata.info()

<class 'pandas.core.frame.DataFrame'>
Index: 773 entries, 0 to 773
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   nombre        773 non-null    string 
 1   municipio     773 non-null    string 
 2   provincia     773 non-null    string 
 3   latitud       773 non-null    float64
 4   longitud      773 non-null    float64
 5   type          771 non-null    string 
 6   tipo-comida   713 non-null    string 
 7   entorno       768 non-null    string 
 8   quality       773 non-null    object 
 9   euskal_sello  773 non-null    object 
 10  email         773 non-null    object 
 11  web           773 non-null    object 
 12  web_euskadi   773 non-null    object 
 13  categoria     773 non-null    object 
 14  calidad       773 non-null    bool   
dtypes: bool(1), float64(2), object(6), string(6)
memory usage: 91.3+ KB


In [20]:
# Los valores ausentes son Python None (no el string "none") → fillna(False)
df_opendata["quality"] = df_opendata["quality"].fillna(False)
df_opendata["euskal_sello"] = df_opendata["euskal_sello"].fillna(False)


In [28]:
df_opendata["euskal_sello"].unique()

array([False, '1'], dtype=object)

In [29]:
# Asignar tipos correctos a todas las columnas
df_opendata["nombre"]     = df_opendata["nombre"].astype("string")
df_opendata["municipio"]  = df_opendata["municipio"].astype("string")
df_opendata["provincia"]  = df_opendata["provincia"].astype("string")
df_opendata["type"]       = df_opendata["type"].astype("string")
df_opendata["tipo-comida"]= df_opendata["tipo-comida"].astype("string")
df_opendata["entorno"]    = df_opendata["entorno"].astype("string")
# map(bool) convierte cualquier valor truthy/falsy antes del cast estricto a boolean
df_opendata["quality"]      = df_opendata["quality"].map(bool).astype("boolean")
df_opendata["euskal_sello"] = df_opendata["euskal_sello"].map(bool).astype("boolean")
df_opendata["email"]      = df_opendata["email"].astype("string")
df_opendata["web"]        = df_opendata["web"].astype("string")
df_opendata["web_euskadi"]= df_opendata["web_euskadi"].astype("string")
df_opendata["categoria"]  = df_opendata["categoria"].astype("string")
df_opendata["calidad"]    = df_opendata["calidad"].astype("boolean")


In [30]:
df_opendata

,nombre,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,quality,euskal_sello,email,web,web_euskadi,categoria,calidad
0,Agorregi,Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",False,False,agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False
1,Aizian,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,False,False,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False
2,Akelarre,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,False,True,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
3,Alameda,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,False,True,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
4,Almiketxu,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,False,True,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
769,Iruri,Legutio,Araba/Álava,42.980472,-2.642584,Tiendas,<NA>,Montes y Valles vascos,False,False,bostibaietasc@gmail.com,https://euskadigastronomika.eus/es/iruri,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False
770,Conservas Arroyabe,Bermeo,Bizkaia,43.422108,-2.745513,Tiendas,<NA>,Costa Vasca,False,False,info@arroyabe.es,https://www.arroyabe.es/,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False
771,Conservas Campos,Bermeo,Bizkaia,43.420379,-2.737545,Tiendas,<NA>,Costa Vasca,False,False,,https://www.clubcampos.com/,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False
772,Burutxaga Hestebeteak,Amurrio,Araba/Álava,43.039381,-3.004257,Tiendas,<NA>,Montes y Valles vascos,False,False,burutxaga.hestebeteak@gmail.com,,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False


# Enriquecer con los datos de Goolge Place

In [32]:

load_dotenv()
API_KEY = os.getenv("API_KEY_MAPS")

#print(API_KEY)

In [ ]:
def search_place_id(nombre, municipio, lat, lng, web, api_key):
    """Busca el Place ID en Google Places. Primero por nombre, luego por web (fallback).
    Devuelve el place_id (str) o None si no se encuentra."""
    location_bias = {"circle": {"center": {"latitude": lat, "longitude": lng}, "radius": 500.0}}

    # Intento 1: por nombre + municipio
    r = requests.post(
        "https://places.googleapis.com/v1/places:searchText",
        json={"textQuery": f"{nombre}, {municipio}", "locationBias": location_bias},
        headers={"Content-Type": "application/json", "X-Goog-Api-Key": api_key, "X-Goog-FieldMask": "places.id"}
    )
    places = (safe_json(r, f"search '{nombre}'") or {}).get('places', [])

    # Intento 2 (fallback): por URL de la web
    if not places and web:
        print(f"[FALLBACK web] '{nombre}' → {web}")
        r2 = requests.post(
            "https://places.googleapis.com/v1/places:searchText",
            json={"textQuery": web, "locationBias": location_bias},
            headers={"Content-Type": "application/json", "X-Goog-Api-Key": api_key, "X-Goog-FieldMask": "places.id"}
        )
        places = (safe_json(r2, f"search web '{nombre}'") or {}).get('places', [])

    return places[0]['id'] if places else None


def get_place_details(place_id, api_key):
    """Obtiene rating, reseñas, precio, teléfono y foto de un Place ID.
    Devuelve el dict de detalles o None si falla."""
    r = requests.get(
        f"https://places.googleapis.com/v1/places/{place_id}",
        headers={
            "X-Goog-Api-Key": api_key,
            "X-Goog-FieldMask": "id,rating,userRatingCount,priceLevel,editorialSummary,nationalPhoneNumber,photos"
        }
    )
    return safe_json(r, f"details '{place_id}'")


def enrich_dataframe(df, api_key, limit=None):
    """Enriquece df_opendata con datos de Google Places API.
    limit=None procesa todas las filas; limit=N procesa solo las primeras N."""
    df["valoracion"]          = np.nan
    df["num_resenas"]         = np.nan
    df["nivel_precio"]        = np.nan
    df["url_imagen"]          = ""
    df["nationalPhoneNumber"] = ""

    subset = df.head(limit) if limit else df
    cont_ok, cont_ko = 0, 0

    for idx, row in subset.iterrows():
        nombre    = row["nombre"]
        municipio = row["municipio"]
        lat, lng  = row["latitud"], row["longitud"]
        web       = row["web"] if pd.notna(row["web"]) and str(row["web"]).strip() else None

        place_id = search_place_id(nombre, municipio, lat, lng, web, api_key)
        if not place_id:
            print(f"[SKIP] sin resultados para '{nombre}'")
            continue

        details = get_place_details(place_id, api_key)
        if not details:
            print(f"[SKIP] sin detalles para '{nombre}'")
            cont_ko += 1
            continue

        df.at[idx, "valoracion"]          = details.get("rating")
        df.at[idx, "num_resenas"]         = details.get("userRatingCount")
        df.at[idx, "nivel_precio"]        = details.get("priceLevel")
        df.at[idx, "nationalPhoneNumber"] = details.get("nationalPhoneNumber", "")

        # Photos API v1: el campo es photos[].name; la URL de media se construye así:
        # https://places.googleapis.com/v1/{photo_name}/media?maxWidthPx=800&key=API_KEY
        photos = details.get("photos", [])
        if photos:
            photo_name = photos[0].get("name", "")
            df.at[idx, "url_imagen"] = (
                f"https://places.googleapis.com/v1/{photo_name}/media?maxWidthPx=800&key={api_key}"
                if photo_name else ""
            )

        cont_ok += 1
        print(f"[OK] '{nombre}' → {details.get('rating')} ⭐ | {details.get('userRatingCount')} reseñas (ok: {cont_ok})")

    print(f"\n--- RESUMEN ---")
    print(f"Enriquecidos: {cont_ok} / {len(subset)} | Fallos en detalles: {cont_ko}")
    return df


# --- Ejecución (prueba con las 10 primeras filas) ---
df_opendata = enrich_dataframe(df_opendata, API_KEY)


C:\Users\jlalo\AppData\Local\Temp\ipykernel_16552\2819010877.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'PRICE_LEVEL_MODERATE' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "nivel_precio"]        = details.get("priceLevel")


[OK] 'Agorregi' → 4.6 ⭐ | 571 reseñas (ok: 1)
[OK] 'Aizian' → 4.7 ⭐ | 435 reseñas (ok: 2)
[OK] 'Akelarre' → 4.5 ⭐ | 1924 reseñas (ok: 3)
[OK] 'Alameda' → 4.6 ⭐ | 1098 reseñas (ok: 4)
[OK] 'Almiketxu' → 4.6 ⭐ | 1924 reseñas (ok: 5)
[OK] 'Ameztoi' → 4.5 ⭐ | 225 reseñas (ok: 6)
[OK] 'Andere' → 4.4 ⭐ | 767 reseñas (ok: 7)
[OK] 'Andra Mari' → 4.7 ⭐ | 1001 reseñas (ok: 8)
[OK] 'Arabako Txakolina, S.L.' → 4.5 ⭐ | 2 reseñas (ok: 9)
[OK] 'Aretxondo' → 4.4 ⭐ | 1059 reseñas (ok: 10)

--- RESUMEN ---
Enriquecidos: 10 / 10 | Fallos en detalles: 0


In [37]:
df_opendata


,nombre,municipio,provincia,latitud,longitud,type,tipo-comida,entorno,quality,euskal_sello,email,web,web_euskadi,categoria,calidad,valoracion,num_resenas,nivel_precio,url_imagen,nationalPhoneNumber
0,Agorregi,Donostia / San Sebastián,Gipuzkoa,43.302405,-2.011846,Restauración,Restaurante,"Costa Vasca,San Sebastián",False,False,agorregi@agorregi.com,https://agorregi.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False,4.6,571.0,PRICE_LEVEL_MODERATE,https://places.googleapis.com/v1/places/ChIJFT...,943 22 43 28
1,Aizian,Bilbao,Bizkaia,43.267519,-2.941807,Restauración,Restaurante,Bilbao,False,False,aizian@restaurante-aizian.com,https://www.restaurante-aizian.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,False,4.7,435.0,PRICE_LEVEL_MODERATE,https://places.googleapis.com/v1/places/ChIJde...,944 28 00 39
2,Akelarre,Donostia / San Sebastián,Gipuzkoa,43.307750,-2.043135,Restauración,Restaurante,San Sebastián,False,True,restaurante@akelarre.net,https://www.akelarre.net,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True,4.5,1924.0,PRICE_LEVEL_VERY_EXPENSIVE,https://places.googleapis.com/v1/places/ChIJXd...,943 31 12 09
3,Alameda,Hondarribia,Gipuzkoa,43.361242,-1.792691,Restauración,Restaurante,Costa Vasca,False,True,info@restaurantealameda.net,https://restaurantealameda.net/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True,4.6,1098.0,PRICE_LEVEL_EXPENSIVE,https://places.googleapis.com/v1/places/ChIJfb...,943 64 27 89
4,Almiketxu,Bermeo,Bizkaia,43.408905,-2.734229,Restauración,Restaurante,Costa Vasca,False,True,almiketxu@gmail.com,https://almiketxu.com/,https://turismoa.euskadi.eus/es/restaurantes/r...,restaurantes,True,4.6,1924.0,PRICE_LEVEL_EXPENSIVE,https://places.googleapis.com/v1/places/ChIJZ5...,946 88 09 25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
769,Iruri,Legutio,Araba/Álava,42.980472,-2.642584,Tiendas,<NA>,Montes y Valles vascos,False,False,bostibaietasc@gmail.com,https://euskadigastronomika.eus/es/iruri,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False,NaN,NaN,NaN,,
770,Conservas Arroyabe,Bermeo,Bizkaia,43.422108,-2.745513,Tiendas,<NA>,Costa Vasca,False,False,info@arroyabe.es,https://www.arroyabe.es/,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False,NaN,NaN,NaN,,
771,Conservas Campos,Bermeo,Bizkaia,43.420379,-2.737545,Tiendas,<NA>,Costa Vasca,False,False,,https://www.clubcampos.com/,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False,NaN,NaN,NaN,,
772,Burutxaga Hestebeteak,Amurrio,Araba/Álava,43.039381,-3.004257,Tiendas,<NA>,Montes y Valles vascos,False,False,burutxaga.hestebeteak@gmail.com,,https://turismoa.euskadi.eus/es/tiendas-gourme...,restaurantes,False,NaN,NaN,NaN,,


In [ ]:
df_opendata.describe(include="all")


,nombre,municipio,provincia,latitud,longitud,organic,type,tipo-comida,entorno,web,categoria
count,774,774,774,774.000000,774.000000,772,772,714,769,774,774
unique,763,175,5,NaN,NaN,3,2,8,11,611,2
top,Txinparta,Donostia / San Sebastián,Gipuzkoa,NaN,NaN,0,Restauración,Restaurante,Montes y Valles vascos,,bodegas
freq,2,59,369,NaN,NaN,768,714,475,339,158,716
mean,NaN,NaN,NaN,43.131510,-2.455699,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,0.255014,0.392737,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,42.488302,-3.386813,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,43.034668,-2.799878,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,43.253863,-2.502752,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,43.307750,-2.095250,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Datos de prueba reales de Open Data Euskadi (ej. un restaurante en Bilbao)
nombre = "Restaurante Zortziko"
municipio = "Bilbao"
lat = 43.2642
lng = -2.9320

location_bias = {
    "circle": {
        "center": {"latitude": lat, "longitude": lng},
        "radius": 500.0
    }
}

print("--- Iniciando prueba de diagnóstico ---")

try:
    response = requests.post(
        "https://places.googleapis.com/v1/places:searchText",
        json={"textQuery": f"{nombre}, {municipio}", "locationBias": location_bias},
        headers={
            "Content-Type": "application/json",
            "X-Goog-Api-Key": API_KEY,
            "X-Goog-FieldMask": "places.id"
        }
    )
    
    print(f"Código de estado HTTP: {response.status_code}")
    print("Respuesta cruda de Google:")
    print(response.text)

except Exception as e:
    print(f"Error al realizar la petición: {e}")

--- Iniciando prueba de diagnóstico ---
Código de estado HTTP: 429
Respuesta cruda de Google:
{
  "error": {
    "code": 429,
    "message": "Quota exceeded for quota metric 'SearchTextRequest' and limit 'SearchTextRequest per day' of service 'places.googleapis.com' for consumer 'project_number:368435468637'.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "RATE_LIMIT_EXCEEDED",
        "domain": "googleapis.com",
        "metadata": {
          "quota_limit_value": "100",
          "quota_unit": "1/d/{project}",
          "service": "places.googleapis.com",
          "quota_limit": "SearchTextRequestPerDayPerProject",
          "quota_location": "global",
          "consumer": "projects/368435468637",
          "quota_metric": "places.googleapis.com/SearchTextRequest"
        }
      },
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            

In [33]:
print(f"Filas a iterar: {len(df_opendata)}")
print(f"Columnas: {list(df_opendata.columns)}")


Filas a iterar: 773
Columnas: ['nombre', 'municipio', 'provincia', 'latitud', 'longitud', 'type', 'tipo-comida', 'entorno', 'quality', 'euskal_sello', 'email', 'web', 'web_euskadi', 'categoria', 'calidad']


In [ ]:
from owslib.wms import WebMapService

# Conectarse al servidor de patrimonio de GeoEuskadi
wms_url = "https://www.geo.euskadi.eus/WMS_KULTURA?request=GetCapabilities&service=WMS"
wms = WebMapService(wms_url, version='1.1.1')

# Ver qué capas de datos de patrimonio están disponibles
print("Capas disponibles en GeoEuskadi - Cultura:")
for layer_name in list(wms.contents):
    print(f"- {layer_name}: {wms[layer_name].title}")

Capas disponibles en GeoEuskadi - Cultura:
- Done_Jakue_bidea_/_Camino_de_Santiago5575: Done Jakue bidea / Camino de Santiago
- Arkeologikoak_/_Arqueologicos27500: Arkeologikoak / Arqueologicos
- Eraikiak_/_Construidos22086: Eraikiak / Construidos
- Ondasun_arkeologikoen_mugaketak_/_Delimitaciones_de_los_bienes_arqueologicos46064: Ondasun arkeologikoen mugaketak / Delimitaciones de los bienes arqueologicos
- Ondasun_arkitekturaen_mugaketak_/_Delimitaciones_de_los_bienes_arquitectonicos58602: Ondasun arkitekturaen mugaketak / Delimitaciones de los bienes arquitectonicos
- Arkeologia-ondarea_/_Patrimonio_arqueologico40227: Arkeologia-ondarea / Patrimonio arqueologico
- Eraikitako_ondarea_/_Patrimonio_construido1888: Eraikitako ondarea / Patrimonio construido


In [ ]:
# Inspeccionar metadatos de una capa concreta
# Cambia 'layer_name' por cualquier nombre de la lista anterior
CAPA = list(wms.contents)[0]   # ← pon aquí el nombre que te interese

layer = wms[CAPA]
print(f"Capa:        {CAPA}")
print(f"Título:      {layer.title}")
print(f"Abstract:    {layer.abstract}")
print(f"CRS:         {layer.crsOptions}")
print(f"BBox (WGS84):{layer.boundingBoxWGS84}")
print(f"Estilos:     {list(layer.styles.keys())}")


Capa:        Done_Jakue_bidea_/_Camino_de_Santiago5575
Título:      Done Jakue bidea / Camino de Santiago
Abstract:    None
CRS:         ['EPSG:3857', 'EPSG:25830', 'EPSG:102100', 'EPSG:4326']
BBox (WGS84):(-19.217266, 27.916894, 5.115361, 47.450573)
Estilos:     ['default']


In [ ]:
from owslib.wfs import WebFeatureService
import geopandas as gpd
from io import BytesIO

# GeoEuskadi usa GeoServer; el WFS tiene un path distinto al WMS
# Candidatos de URL ordenados por probabilidad
WFS_CANDIDATES = [
    "https://www.geo.euskadi.eus/geoeuskadi/services/WFS_KULTURA/GeoServer/wfs",
    "https://www.geo.euskadi.eus/geoeuskadi/services/KULTURA_WFS/GeoServer/wfs",
    "https://www.geo.euskadi.eus/geoeuskadi/services/WFS_KULTURA/wfs",
]

wfs = None
for url in WFS_CANDIDATES:
    try:
        wfs = WebFeatureService(url, version='2.0.0')
        print(f"[OK] WFS encontrado en: {url}")
        break
    except Exception as e:
        print(f"[FAIL] {url} → {e}")

if wfs:
    print("\nCapas WFS disponibles:")
    for name in list(wfs.contents):
        print(f"  {name}: {wfs[name].title}")
else:
    print("\n[INFO] WFS no disponible para KULTURA.")
    print("Alternativa: usar la API REST de GeoEuskadi o el catálogo de Open Data Euskadi.")
    print("Catálogo: https://opendata.euskadi.eus/catalogo/-/patrimonio-cultural/")


[FAIL] https://www.geo.euskadi.eus/geoeuskadi/services/WFS_KULTURA/GeoServer/wfs → 404 Client Error: Not Found for url: https://www.geo.euskadi.eus/geoeuskadi/services/WFS_KULTURA/GeoServer/wfs?service=WFS&request=GetCapabilities&version=2.0.0
[FAIL] https://www.geo.euskadi.eus/geoeuskadi/services/KULTURA_WFS/GeoServer/wfs → 404 Client Error: Not Found for url: https://www.geo.euskadi.eus/geoeuskadi/services/KULTURA_WFS/GeoServer/wfs?service=WFS&request=GetCapabilities&version=2.0.0
[FAIL] https://www.geo.euskadi.eus/geoeuskadi/services/WFS_KULTURA/wfs → 404 Client Error: Not Found for url: https://www.geo.euskadi.eus/geoeuskadi/services/WFS_KULTURA/wfs?service=WFS&request=GetCapabilities&version=2.0.0

[INFO] WFS no disponible para KULTURA.
Alternativa: usar la API REST de GeoEuskadi o el catálogo de Open Data Euskadi.
Catálogo: https://opendata.euskadi.eus/catalogo/-/patrimonio-cultural/


In [ ]:
if wfs is None:
    print("[INFO] WFS no disponible — no se pueden descargar features.")
    print("Usa load_geojson_jsonp() con las URLs del catálogo de Open Data Euskadi:")
    print("  https://opendata.euskadi.eus/catalogo/-/patrimonio-cultural/")
else:
    # Descargar los features de una capa WFS como GeoDataFrame
    CAPA_WFS = list(wfs.contents)[0]  # ← cambia por la capa que te interese
    print(f"Descargando capa: {CAPA_WFS}")

    response = wfs.getfeature(
        typename=CAPA_WFS,
        outputFormat="application/json",
        maxfeatures=100    # límite para exploración; quita para obtener todos
    )

    gdf = gpd.read_file(BytesIO(response.read()))
    print(f"Features descargados: {len(gdf)}")
    print(f"Columnas: {list(gdf.columns)}")
    display(gdf.head())


[INFO] WFS no disponible — no se pueden descargar features.
Usa load_geojson_jsonp() con las URLs del catálogo de Open Data Euskadi:
  https://opendata.euskadi.eus/catalogo/-/patrimonio-cultural/
